# Suite VLG — The Verilog language

Verilog and SystemVerilog as one tree language (`@mbse/programs/Verilog/Syntax`): its kinds and categories,
organized as IEEE 1800's grammar is, the identifiers it accepts, and where each kind and feature exists in the two
families, which the standards check.

In [ ]:
import { SystemVerilog2017, SystemVerilog2023, Syntax as S, Verilog2005, VerilogStandard } from "@mbse/programs/Verilog";

import { assert, equal } from "./support.js";

const L = S.LANGUAGE;
const [V1995, V2001] = [new VerilogStandard(1995, "Verilog"), new VerilogStandard(2001, "Verilog")];
const [SV2005, SV2009, SV2012] = [new VerilogStandard(2005), new VerilogStandard(2009), new VerilogStandard(2012)];
const name = (spelling: unknown) => new S.Identifier({ spelling });
const ref = (spelling: string) => new S.NameExpression({ name: name(spelling) });
const number = (spelling = "1") => new S.IntegerLiteral({ spelling });
const unit = (...items: any[]) => new S.SourceText({ items });
const module = (items: any[] = [], properties: object = {}) =>
  new S.ModuleDeclaration({ keyword: "module", name: name("m"), items, ...properties });

## VLG-01 · Every kind belongs to one category, and the grammar lists them

In [ ]:
const kinds = L.kinds();
assert(kinds.size === S.KINDS.length && S.KINDS.length === 202);
equal([...kinds.keys()], S.KINDS.map((k) => k.name));
equal([...L.categories().keys()].sort(), ["Name", "Item", "Port", "DataType", "Dimension", "Statement", "Connection",
  "TimingControl", "Expression", "Literal", "Range", "Directive", "Constraint", "Property", "Sequence", "BinsSelect"].sort());
assert(new S.IntegerLiteral() instanceof S.Literal && new S.IntegerLiteral() instanceof S.Expression);
assert(S.KINDS.every((k) => k.NAME === `Programs.Verilog.${k.KIND}`));
const grammar = L.grammar() as any;
equal(grammar.base, { Verilog: 1995, SystemVerilog: 2005 });
assert(grammar.kinds.length === 202);
equal([S.sv(), S.sv(2009), S.verilog(2001)], [{ SystemVerilog: 2005 }, { SystemVerilog: 2009 },
  { Verilog: 2001, SystemVerilog: 2005 }]);
assert(new S.Comment({ text: " x" }).block === false && new S.SourceText().items.length === 0);
console.log("ok");

## VLG-02 · Identifiers are simple or escaped, and system names start with `$`

In [ ]:
equal(L.validate(name("x_1$")), []);
equal(L.validate(name("\\bus+index")), []);
equal(L.validate(name("")), []);
for (const bad of ["1x", "$x", "a b", "Größe", "\\", "\\a b"]) {
  equal(L.validate(name(bad)), [`${bad === "\\" ? "'\\\\'" : bad === "\\a b" ? "'\\\\a b'" : `'${bad}'`} is not an identifier`]);
}
equal(L.validate(name(3n)), ["Identifier.spelling must be a str, got int"]);
equal(L.validate(new S.SystemCall({ name: "$clog2" })), []);
equal(L.validate(new S.SystemCall({ name: "" })), []);
for (const bad of ["clog2", "$", "$a b", "$é"]) {
  equal(L.validate(new S.SystemCall({ name: bad })), [`'${bad}' is not a system task or function name`]);
}
equal(L.validate(new S.SystemCall({ name: 1n })), ["SystemCall.name must be a str, got int"]);
console.log("ok");

## VLG-03 · Kinds and features exist by family and year: Verilog has none of SystemVerilog's own

In [ ]:
equal([Verilog2005.STANDARD.name(), SystemVerilog2017.STANDARD.name(), SystemVerilog2023.STANDARD.name(), V1995.name()],
  ["Verilog-2005", "SystemVerilog-2017", "SystemVerilog-2023", "Verilog-1995"]);
const pkg = new S.PackageDeclaration({ name: name("p") });
equal(Verilog2005.check(pkg), ["PackageDeclaration is not Verilog"]);
equal(SV2005.check(pkg), []);
const ansi = new S.AnsiPort({ direction: "input", name: name("a") });
equal(V1995.check(ansi), ["AnsiPort needs Verilog-2001"]);
equal(V2001.check(ansi), []);
equal(Verilog2005.check(new S.IntegerVectorType({ keyword: "logic" })), ["IntegerVectorType.keyword 'logic' is not Verilog"]);
equal(Verilog2005.check(new S.IntegerVectorType({ keyword: "reg", signing: "signed" })), []);
equal(V1995.check(new S.IntegerVectorType({ keyword: "reg", signing: "signed" })),
  ["IntegerVectorType.signing 'signed' needs Verilog-2001"]);
const unique0 = new S.IfStatement({ qualifier: "unique0", condition: ref("a"), consequence: new S.NullStatement() });
equal(SV2005.check(unique0), ["IfStatement.qualifier 'unique0' needs SystemVerilog-2009"]);
equal(SV2009.check(unique0), []);
const final = new S.ImmediateAssertion({ keyword: "assert", deferral: "final", expression: ref("a") });
equal(SV2009.check(final), ["ImmediateAssertion.deferral 'final' needs SystemVerilog-2012"]);
equal(SV2012.check(final), []);
const star = new S.EventControl();
equal(V1995.check(star), ["EventControl * needs Verilog-2001"]);
equal(V2001.check(star), []);
equal(Verilog2005.check(new S.DisableStatement()), ["DisableStatement fork is not Verilog"]);
equal(Verilog2005.check(new S.DisableStatement({ target: ref("b") })), []);
const declared = new S.ForStatement({ initializers: [new S.VariableDeclaration({
  type: new S.IntegerAtomType({ keyword: "integer" }),
  declarators: [new S.VariableDeclarator({ name: name("i"), value: number() })] })], body: new S.NullStatement() });
equal(Verilog2005.check(declared), ["ForStatement.initializers declarations is not Verilog"]);
const assign = (target: string) => new S.AssignmentExpression({ target: ref(target), operator: "=", value: number() });
const two = new S.ForStatement({ initializers: [assign("i"), assign("j")], body: new S.NullStatement() });
equal(Verilog2005.check(two), ["ForStatement.initializers declarations is not Verilog"]);
equal(SV2005.check(two), []);
equal(Verilog2005.check(new S.ForStatement({ initializers: [assign("i")], body: new S.NullStatement() })), []);
equal(Verilog2005.check(module([], { labeled: true })), ["ModuleDeclaration.labeled is not Verilog"]);
equal(V1995.check(new S.BinaryExpression({ left: number(), operator: "**", right: number() })),
  ["BinaryExpression.operator '**' needs Verilog-2001"]);
equal(SV2005.check(new S.BinaryExpression({ left: number(), operator: "<->", right: number() })),
  ["BinaryExpression.operator '<->' needs SystemVerilog-2009"]);
const interfaceClass = new S.ClassDeclaration({ interface: true, name: name("i") });
equal(Verilog2005.check(interfaceClass), ["ClassDeclaration is not Verilog", "ClassDeclaration.interface is not Verilog"]);
equal(SV2009.check(interfaceClass), ["ClassDeclaration.interface needs SystemVerilog-2012"]);
equal(SV2009.check(new S.ForwardTypedefDeclaration({ keyword: "interface class", name: name("i") })),
  ["ForwardTypedefDeclaration.keyword 'interface class' needs SystemVerilog-2012"]);
equal(Verilog2005.check(new S.VariableDeclaration({ random: "rand", declarators: [new S.VariableDeclarator({ name: name("r") })] })),
  ["VariableDeclaration.random is not Verilog"]);
equal(Verilog2005.check(new S.NewExpression()), ["NewExpression is not Verilog"]);
console.log("ok");

## VLG-04 · A standard checks a whole tree, by path

In [ ]:
const tree = unit(module([new S.AlwaysConstruct({ keyword: "always_ff", body: new S.NullStatement() }),
  new S.ContinuousAssign({ assignments: [new S.AssignmentExpression({ target: ref("a"), operator: "+=", value: number() })] })]));
equal(Verilog2005.check(tree), ["items[0].items[0]: AlwaysConstruct.keyword 'always_ff' is not Verilog",
  "items[0].items[1].assignments[0]: AssignmentExpression.operator '+=' is not Verilog"]);
equal(SystemVerilog2023.check(tree), []);
console.log("ok");

## VLG-05 · Classes and methods check their own shape

In [ ]:
const method = (kind: typeof S.FunctionDeclaration | typeof S.TaskDeclaration, properties: object = {}) =>
  new kind({ name: name("f"), ...properties });
for (const kind of [S.FunctionDeclaration, S.TaskDeclaration]) {
  equal(L.validate(method(kind, { virtual: true, pure: true })), []);
  equal(L.validate(method(kind, { extern: true })), []);
  equal(L.validate(method(kind, { pure: true })), [`a pure ${kind.KIND} is virtual`]);
  equal(L.validate(method(kind, { extern: true, body: [new S.NullStatement()] })), [`an extern or pure ${kind.KIND} has no body`]);
  equal(L.validate(method(kind, { virtual: true, pure: true, labeled: true })), [`an extern or pure ${kind.KIND} has no body`]);
}
const base = () => new S.NamedType({ name: name("b") });
equal(L.validate(new S.ClassDeclaration({ name: name("c"), base: base(), arguments: [number()] })), []);
equal(L.validate(new S.ClassDeclaration({ interface: true, name: name("i"), interfaces: [new S.NamedType({ name: name("j") })] })), []);
equal(L.validate(new S.ClassDeclaration({ interface: true, name: name("i"), base: base() })),
  ["an interface ClassDeclaration has no virtual, base or arguments"]);
equal(L.validate(new S.ClassDeclaration({ interface: true, virtual: true, name: name("i") })),
  ["an interface ClassDeclaration has no virtual, base or arguments"]);
equal(L.validate(new S.ClassDeclaration({ name: name("c"), arguments: [number()] })), ["a ClassDeclaration with arguments has a base"]);
console.log("ok");

## VLG-06 · Distributions check their own shape, and where constraints and randomization exist

In [ ]:
equal(L.validate(new S.DistItem({ value: number() })), []);
equal(L.validate(new S.DistItem({ value: number(), operator: ":=", weight: number() })), []);
for (const item of [new S.DistItem({ value: number(), operator: ":=" }), new S.DistItem({ value: number(), weight: number() })]) {
  equal(L.validate(item), ["a DistItem has both an operator and a weight, or neither"]);
}
equal(L.validate(new S.DistItem({ operator: ":=", weight: number() })), ["a default DistItem weighs its values with :/"]);
equal(L.validate(new S.DistItem()), ["a default DistItem weighs its values with :/"]);
equal(L.validate(new S.DistItem({ operator: ":/", weight: number() })), []);
equal(SystemVerilog2017.check(new S.DistItem({ operator: ":/", weight: number() })), ["DistItem default needs SystemVerilog-2023"]);
equal(SV2009.check(new S.ExpressionConstraint({ soft: true, expression: ref("a") })),
  ["ExpressionConstraint.soft needs SystemVerilog-2012"]);
equal(SV2009.check(new S.ConstraintPrototype({ qualifier: "pure", name: name("k") })),
  ["ConstraintPrototype.qualifier 'pure' needs SystemVerilog-2012"]);
equal(SV2009.check(new S.UniqueConstraint({ set: [ref("a")] })), ["UniqueConstraint needs SystemVerilog-2012"]);
equal(SV2005.check(new S.LocalName({ name: name("x") })), ["LocalName needs SystemVerilog-2009"]);
equal(SV2009.check(new S.RandomizeWithExpression({ call: ref("r"), restricted: true })),
  ["RandomizeWithExpression.restricted needs SystemVerilog-2012"]);
equal(Verilog2005.check(new S.RandCaseStatement()), ["RandCaseStatement is not Verilog"]);
console.log("ok");

## VLG-07 · Where assertions, their properties and clocking exist

In [ ]:
const restrict = new S.ConcurrentAssertion({ keyword: "restrict", spec: new S.PropertySpec({ property: ref("a") }) });
equal(SV2005.check(restrict), ["ConcurrentAssertion.keyword 'restrict' needs SystemVerilog-2009"]);
equal(SV2009.check(restrict), []);
equal(Verilog2005.check(restrict), ["ConcurrentAssertion is not Verilog", "ConcurrentAssertion.keyword 'restrict' is not Verilog",
  "spec: PropertySpec is not Verilog"]);
const since2009 = [new S.ImplicationProperty({ antecedent: ref("a"), operator: "#-#", consequent: ref("b") }),
  new S.BinaryProperty({ left: ref("a"), operator: "until", right: ref("b") }),
  new S.UnaryProperty({ operator: "always", operand: ref("a") }), new S.StrengthProperty({ keyword: "strong", sequence: ref("a") }),
  new S.AbortProperty({ keyword: "accept_on", condition: ref("a"), operand: ref("b") }), new S.CaseProperty({ expression: ref("a") }),
  new S.LetDeclaration({ name: name("l"), value: ref("a") }), new S.DefaultDisable({ condition: ref("a") }),
  new S.ClockingDeclaration({ scope: "global", clock: new S.EventControl() }), new S.AssertionPort({ local: true, name: name("p") }),
  new S.KeywordType({ keyword: "untyped" })];
for (const node of since2009) {
  const problems = SV2005.check(node);
  assert(problems.length > 0 && problems.every((p) => p.includes("needs SystemVerilog-2009")) && SV2009.check(node).length === 0,
    String(node));
}
equal(SV2005.check(new S.UnaryProperty({ operator: "not", operand: ref("a") })), []);
equal(SV2005.check(new S.ImplicationProperty({ antecedent: ref("a"), operator: "|->", consequent: ref("b") })), []);
console.log("ok");

## VLG-08 · Covergroups, coverpoints, bins and transitions check their own shape, and where they exist

In [ ]:
const clock = new S.EventControl({ events: [new S.EventExpression({ expression: ref("clk") })] });
equal(L.validate(new S.CovergroupDeclaration({ name: name("g"), clock, sample: new S.SampleFunction() })),
  ["a CovergroupDeclaration has a clock or a sample, not both"]);
equal(L.validate(new S.Coverpoint({ type: new S.IntegerAtomType({ keyword: "int" }), expression: ref("a") })),
  ["a Coverpoint with a type has a label"]);
equal(L.validate(new S.CoverageBins({ keyword: "bins", name: name("b"), size: number(), initializer: new S.BinsDefault() })),
  ["a CoverageBins with a size is an array"]);
equal(L.validate(new S.CoverageBins({ keyword: "bins", name: name("b"), array: true, size: number(),
  initializer: new S.BinsDefault() })), []);
equal(L.validate(new S.TransitionStep({ values: [number()], operator: "*" })),
  ["a TransitionStep has both an operator and a count, or neither"]);
equal(L.validate(new S.TransitionStep({ values: [number()], operator: "*", count: number() })), []);
equal(Verilog2005.check(new S.CovergroupDeclaration({ name: name("g") })), ["CovergroupDeclaration is not Verilog"]);
const since2012 = [new S.BinsExpression({ expression: ref("a") }), new S.FilteredBinsSelect({ select: ref("a"), filter: ref("b") }),
  new S.BinsValues({ values: [number()], filter: ref("item") }),
  new S.Coverpoint({ label: name("c"), type: new S.IntegerAtomType({ keyword: "int" }), expression: ref("a") })];
for (const node of since2012) {
  const problems = SV2009.check(node);
  assert(problems.length > 0 && problems.every((p) => p.includes("needs SystemVerilog-2012")) && SV2012.check(node).length === 0,
    String(node));
}
equal(SV2005.check(new S.SampleFunction()), ["SampleFunction needs SystemVerilog-2009"]);
console.log("ok");

## VLG-09 · Attributes exist from Verilog-2001

In [ ]:
const keep = [new S.AttributeInstance({ specs: [new S.AttributeSpec({ name: name("keep") })] })];
const keptWire = new S.NetDeclaration({ net_type: "wire", declarators: [new S.VariableDeclarator({ name: name("w") })] });
const attributedWire = new S.AttributedItem({ attributes: keep, item: keptWire });
equal(V1995.check(attributedWire), ["AttributedItem needs Verilog-2001", "attributes[0]: AttributeInstance needs Verilog-2001",
  "attributes[0].specs[0]: AttributeSpec needs Verilog-2001"]);
equal(V2001.check(attributedWire), []);
for (const node of [new S.BinaryExpression({ left: ref("a"), operator: "+", attributes: [new S.AttributeInstance()], right: ref("b") }),
  new S.UnaryExpression({ operator: "-", attributes: [new S.AttributeInstance()], operand: ref("a") }),
  new S.ConditionalExpression({ condition: ref("a"), attributes: [new S.AttributeInstance()], consequence: ref("b"),
    alternative: ref("c") }),
  new S.CallExpression({ callee: ref("f"), attributes: [new S.AttributeInstance()] }),
  new S.SystemCall({ name: "$f", attributes: [new S.AttributeInstance()] })]) {
  const problems = V1995.check(node);
  assert(problems.includes(`${node.kind().KIND}.attributes needs Verilog-2001`) && V2001.check(node).length === 0,
    problems.join("; "));
}
console.log("ok");